In [57]:
from cassandra.cluster import Cluster
from cassandra.query import SimpleStatement
from cassandra import ConsistencyLevel

cluster = Cluster(['localhost'], port=9042)
session = cluster.connect('leaderboards')


In [58]:
import pandas as pd

## Lecturas

### Hall of Fame

In [59]:
paises = pd.read_csv('./csv_tablas/dungeons_by_country.csv')

In [60]:
paises.country.unique()

<StringArray>
['ja_JP', 'en_US', 'fr_FR', 'ko_KR', 'pt_BR', 'it_IT', 'es_ES', 'de_DE',
 'ru_RU', 'zh_CN', 'zh_TW']
Length: 11, dtype: str

In [61]:
def id_dungeons_of_country(session, country):
    query = "SELECT dungeon_id FROM dungeons_by_country WHERE country = %s;"
    statement = SimpleStatement(query, consistency_level=ConsistencyLevel.QUORUM)
    resultados = session.execute(statement, [country])
    
    dungeons_id = []

    for fila in resultados:
        dungeons_id.append(fila.dungeon_id)

    return dungeons_id

In [62]:
id_dungeons_of_country(session, 'es_ES')

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

In [69]:
def top_by_dungeon_and_country(session, country, dungeon_id, k=5):
    query = "SELECT dungeon_id, dungeon_name, time_minutes, user_name, email, date FROM hall_of_fame_by_country WHERE country = %s AND dungeon_id = %s LIMIT %s;"
    statement = SimpleStatement(query, consistency_level=ConsistencyLevel.QUORUM)
    resultados = session.execute(statement, [country, dungeon_id, k])
    
    dungeon_id = resultados[0].dungeon_id
    dungeon_name = resultados[0].dungeon_name

    top = []

    for fila in resultados:
        top.append({'email': fila.email, 'user_name': fila.user_name, 'time_minutes': fila.time_minutes, 'date': fila.date.isoformat()})

    return {'dungeon_id': dungeon_id, 'dungeon_name': dungeon_name, f'top_{k}': top}


In [70]:
top_by_dungeon_and_country(session, 'es_ES', 1, 5)

{'dungeon_id': 1,
 'dungeon_name': 'Burgstream, Culverts of the Bashful Sumo Wrestlers',
 'top_5': [{'email': 'abellanjulio@example.org',
   'user_name': 'garciamartirio',
   'time_minutes': 0,
   'date': '2018-12-04T02:45:50'},
  {'email': 'agulloricarda@example.org',
   'user_name': 'angelino53',
   'time_minutes': 0,
   'date': '2012-10-08T01:21:09'},
  {'email': 'amadorbarbero@example.com',
   'user_name': 'sbarba',
   'time_minutes': 0,
   'date': '2022-10-10T03:12:30'},
  {'email': 'berta74@example.com',
   'user_name': 'aparicioromulo',
   'time_minutes': 0,
   'date': '2016-09-26T04:29:39'},
  {'email': 'brionesjose-antonio@example.com',
   'user_name': 'brumaricruz',
   'time_minutes': 0,
   'date': '2022-03-11T00:22:35'}]}

In [74]:
def hall_of_fame(session, country):
    
    dungeon_ids = id_dungeons_of_country(session, country)

    tops_pais = []

    for dungeon_id in dungeon_ids:
        tops_pais.append(top_by_dungeon_and_country(session, country, dungeon_id, 5))
        
    return tops_pais


In [75]:
hall_of_fame(session, 'es_ES')

[{'dungeon_id': 0,
  'dungeon_name': 'Burghap, Prison of the Jealous Hippies',
  'top_5': [{'email': 'begonavera@example.net',
    'user_name': 'victorino97',
    'time_minutes': 0,
    'date': '2019-12-17T03:26:00'},
   {'email': 'cbarrena@example.net',
    'user_name': 'cecilia28',
    'time_minutes': 0,
    'date': '2019-10-15T15:11:14'},
   {'email': 'macario68@example.com',
    'user_name': 'julianesther',
    'time_minutes': 0,
    'date': '2022-03-25T05:52:29'},
   {'email': 'bllabres@example.org',
    'user_name': 'bibianaparedes',
    'time_minutes': 1,
    'date': '2020-10-03T13:15:35'},
   {'email': 'candido53@example.net',
    'user_name': 'jesustormo',
    'time_minutes': 1,
    'date': '2022-09-19T18:53:03'}]},
 {'dungeon_id': 1,
  'dungeon_name': 'Burgstream, Culverts of the Bashful Sumo Wrestlers',
  'top_5': [{'email': 'abellanjulio@example.org',
    'user_name': 'garciamartirio',
    'time_minutes': 0,
    'date': '2018-12-04T02:45:50'},
   {'email': 'agulloricarda@ex

## Escritura